# Text Generation with Hugging Face Transformers 🤗

This notebook explores various ways to generate text using the `transformers` library. We will cover high-level pipelines, direct model/tokenizer interaction, tokenization details, and different decoding strategies.

## Environment Setup
Before starting, ensure you have the necessary environment and dependencies:
- **Environment**: `conda activate hugvenv312`
- **Installation**: `pip install -r requirements.txt`

### Useful Resources
- [Transformer Explainer](https://poloclub.github.io/transformer-explainer/): An interactive visualization of how Transformers work.
- [Hugging Face Documentation](https://huggingface.co/docs/transformers/index): Official documentation for the `transformers` library.

In [1]:
import warnings

# Suppress warnings for a cleaner output
warnings.filterwarnings("ignore")

print("Jupyter Notebook Initialized: Hugging Face Transformers Exploration")

Jupyter Notebook Initialized: Hugging Face Transformers Exploration


In [2]:
# System Diagnostics: Check CPU, GPU, and Library Versions
import platform
import torch
import torch.nn.functional as F
import subprocess

print(f"--- System Information ---")
print(f"Platform: {platform.platform()}")
print(f"Python:   {platform.python_version()}")
print(f"PyTorch:  {torch.__version__}")

print(f"\n--- GPU/ROCm Accelerators ---")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"Device Count:   {torch.cuda.device_count()}")
    print(f"Primary Device: {torch.cuda.get_device_name(0)}")
else:
    print("Optimization Note: No CUDA-capable GPU detected. Operations will run on CPU.")

--- System Information ---
Platform: Windows-11-10.0.26200-SP0
Python:   3.12.13
PyTorch:  2.9.1+rocm7.2.1

--- GPU/ROCm Accelerators ---
CUDA Available: True
Device Count:   1
Primary Device: AMD Radeon RX 7900 XT


## Method 1: Using the High-Level `pipeline` API

The `pipeline` is the easiest way to use a model for a specific task. It abstracts away the complexity of tokenization, model inference, and decoding.

We'll use GPT-2 for text generation.

In [3]:
from transformers import pipeline

# Initialize the text-generation pipeline with GPT-2
# This automatically downloads the model and tokenizer if not already present
generator = pipeline('text-generation', model='openai-community/gpt2')

# Define a prompt
prompt = 'What is machine learning?'

# Generate text
# The pipeline returns a list of dictionaries containing the generated text
results = generator(prompt, max_length=50, num_return_sequences=1)

# Print the result
print(results[0]['generated_text'])

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is machine learning?

This is a very complicated question, and there are many different aspects of machine learning (e.g., deep learning, machine learning, etc.), but machine learning has been a great approach to solving problems and has been


## Method 2: Manual Control with Models and Tokenizers

For more granular control, you can load the tokenizer and model independently. This allows you to inspect intermediate steps like token IDs and attention masks.

### Loading Model & Tokenizer
We use `AutoTokenizer` and `AutoModelForCausalLM` which automatically select the correct classes based on the model checkpoint.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'openai-community/gpt2'

# Load the tokenizer responsible for converting text to numbers
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the causal language model
model = AutoModelForCausalLM.from_pretrained(model_id)

print(f"Tokenizer loaded: {type(tokenizer).__name__}")
print(f"Model loaded:     {type(model).__name__}")

Tokenizer loaded: GPT2TokenizerFast
Model loaded:     GPT2LMHeadModel


In [5]:
# Convert text to token IDs
sentence = 'What is machine learning?'
inputs = tokenizer(sentence, return_tensors='pt')
input_ids = inputs["input_ids"]

print(f"Input IDs: {input_ids}")

# Convert back to text
decoded_text = tokenizer.decode(input_ids[0])
print(f"Decoded:   {decoded_text}")

Input IDs: tensor([[2061,  318, 4572, 4673,   30]])
Decoded:   What is machine learning?


### Understanding Subword Tokenization

Modern tokenizers split words into "subwords" or "tokens". This allows the model to handle a large vocabulary with a smaller set of base units. 

Let's look at how the word `unbelievable` is decomposed.

In [6]:
word = 'unbelievable'
input_ids = tokenizer(word, return_tensors='pt').input_ids

# Show the IDs and the reconstructed word
print(f"Token IDs: {input_ids}")
print(f"Decoded:   {tokenizer.decode(input_ids[0])}")

Token IDs: tensor([[  403,  6667, 11203,   540]])
Decoded:   unbelievable


In [7]:
# Inspect individual tokens
for token_id in input_ids[0]:
    print(f"ID {token_id:5} -> '{tokenizer.decode(token_id)}'")

ID   403 -> 'un'
ID  6667 -> 'bel'
ID 11203 -> 'iev'
ID   540 -> 'able'


### Handling Rare Words (Out-of-Vocabulary)

When a word is rare (like `homoscendasticity`), the tokenizer breaks it down into even smaller phonetic or character-level fragments. This ensures every possible string can be represented.

In [8]:
complex_word = 'homoscendasticity'
ids = tokenizer(complex_word, return_tensors='pt').input_ids

print(f"Breakdown for '{complex_word}':")
for token_id in ids[0]:
    print(f"ID {token_id:5} -> '{tokenizer.decode(token_id)}'")

full_decoded = tokenizer.decode(ids.squeeze())
print(f"\nReassembled: {full_decoded}")

Breakdown for 'homoscendasticity':
ID 26452 -> 'hom'
ID   418 -> 'os'
ID 15695 -> 'cend'
ID  3477 -> 'astic'
ID   414 -> 'ity'

Reassembled: homoscendasticity


## Probability Distributions and Logits

Language models predict the next token by outputting "logits" (raw scores) for every token in their vocabulary. These scores are then converted into probabilities.

Let's see what the model thinks should come next in a sentence.

In [9]:
# Get model outputs (logits) for the last token in the sequence
with torch.no_grad():
    outputs = model(input_ids)
    
last_token_logits = outputs.logits[0, -1]

# Find the most likely next token ID
predicted_id = last_token_logits.argmax()
predicted_word = tokenizer.decode(predicted_id)

print(f"Context:        '{tokenizer.decode(input_ids[0])}'")
print(f"Most likely:    '{predicted_word}'")

Context:        'unbelievable'
Most likely:    '"'


## Text Generation Strategies (Decoding)

There are several ways to choose the "next" token from the probability distribution. Each has trade-offs between accuracy and creativity:

1.  **Greedy Decoding**: Always pick the single most probable token. (Deterministic but can be repetitive).
2.  **Top-K Sampling**: Randomly sample from the top `k` probable tokens.
3.  **Top-P (Nucleus) Sampling**: Randomly sample from the smallest set of tokens whose cumulative probability $\ge p$.
4.  **Temperature Sampling**: Scaling the distribution before sampling. High temperature (>1) = more random; Low temperature (<1) = more confident.
5.  **Random Sampling**: Sample from the entire vocabulary based on the probabilities. (Highly unpredictable).

In [10]:
def greedy_decode(logits):
    """Selects the absolute most likely next token."""
    return torch.argmax(logits, dim=-1)

def top_k_sampling(logits, k=50):
    """Filters for the top-k most likely tokens, then samples from them."""
    values, indices = torch.topk(logits, k=k)
    probs = F.softmax(values, dim=-1)
    sampled_index = torch.multinomial(probs, num_samples=1)
    return indices[sampled_index]

def top_p_sampling(logits, p=0.9):
    """Samples from the smallest set of tokens that account for p% of the probability mass."""
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Mask tokens outside the nucleus
    mask = cumulative_probs > p
    # Shift mask to include the first token that exceeds p
    mask[..., 1:] = mask[..., :-1].clone()
    mask[..., 0] = False
    
    sorted_logits[mask] = float('-inf')
    probs = F.softmax(sorted_logits, dim=-1)
    sampled_index = torch.multinomial(probs, num_samples=1)
    return sorted_indices[sampled_index]

def temperature_sampling(logits, temperature=1.0):
    """Distorts the distribution to increase or decrease randomness."""
    adjusted_logits = logits / max(temperature, 1e-5) # Avoid division by zero
    probs = F.softmax(adjusted_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

def random_sampling(logits):
    """Plain multinomial sampling from the full distribution."""
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

In [11]:
# Demonstrate different decoding predictions for a single prompt
prompt = 'Today I decided to go to the local'
inputs = tokenizer(prompt, return_tensors='pt').input_ids
with torch.no_grad():
    logits = model(inputs).logits[0, -1]

print(f"Prompt: '{prompt} ...'\n")
print(f"Greedy:      '{tokenizer.decode(greedy_decode(logits))}'")
print(f"Top-K:       '{tokenizer.decode(top_k_sampling(logits, k=5))}'")
print(f"Top-P:       '{tokenizer.decode(top_p_sampling(logits, p=0.8))}'")
print(f"Temp (1.5):  '{tokenizer.decode(temperature_sampling(logits, temperature=1.5))}'")
print(f"Random:      '{tokenizer.decode(random_sampling(logits))}'")

Prompt: 'Today I decided to go to the local ...'

Greedy:      ' library'
Top-K:       ' library'
Top-P:       ' school'
Temp (1.5):  ' porn'
Random:      ' papers'


In [12]:
# Visualizing the Probability Distribution
prompt = 'Machine learning is becoming very'
inputs = tokenizer(prompt, return_tensors='pt').input_ids

with torch.no_grad():
    logits = model(inputs).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)

# Get the top 10 most likely candidates
top_values, top_indices = torch.topk(probs, k=10)

print(f"Top 10 predictions for: '{prompt} ...'\n")
for val, idx in zip(top_values, top_indices):
    token_str = tokenizer.decode([idx])
    print(f"{token_str:15} | Probability: {val.item():.2%}")

Top 10 predictions for: 'Machine learning is becoming very ...'

 popular        | Probability: 28.36%
 powerful       | Probability: 6.24%
 important      | Probability: 5.42%
 complex        | Probability: 3.46%
 much           | Probability: 3.39%
 useful         | Probability: 2.63%
 sophisticated  | Probability: 2.27%
 difficult      | Probability: 2.17%
 common         | Probability: 1.80%
 interesting    | Probability: 1.42%


## Introduction to Datasets for Text Analysis

Hugging Face provides the `datasets` library to easily load and manage large-scale data for NLP tasks. Here, we'll look at the IMDB Movie Reviews dataset, commonly used for Sentiment Analysis.

In [13]:
from datasets import load_dataset

# Load the IMDB dataset (automatically caches locally)
dataset = load_dataset("stanfordnlp/imdb")

print(f"Dataset type: {type(dataset)}")
print(dataset)

Dataset type: <class 'datasets.dataset_dict.DatasetDict'>
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [20]:
import pandas as pd

# Convert the training split to a Pandas DataFrame for easier exploration
my_dataset_df = dataset['train'].to_pandas()

# Display the first few rows
print(f"Total rows in training set: {len(my_dataset_df)}")
my_dataset_df.head()

Total rows in training set: 25000


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [ ]:
from transformers import pipeline

# Default model for sentiment analysis fine-tuned on the SST-2 dataset (model='distilbert-base-uncased-finetuned-sst-2-english')
classifier = pipeline('text-classification', model='distilbert-base-uncased-finetuned-sst-2-english')

print("text : this day is great!")
print(classifier('this day is great!'))
print(" ")
print("text : this day is terrible and I am very sad about it.")
print(classifier('this day is terrible and I am very sad about it.'))

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


text : this day is great!
[{'label': 'POSITIVE', 'score': 0.999881386756897}]
text : this day is terrible and I am very sad about it.
[{'label': 'NEGATIVE', 'score': 0.9989134073257446}]


In [ ]:
def score(review_text):
    """Returns the sentiment score for a given review text."""
    # We truncate to 500 characters as requested
    return classifier(review_text[:500])[0]["label"]

# Apply the score function to the dataset
my_dataset_df["model_prediction"] = my_dataset_df["text"].apply(score)

In [ ]:
my_dataset_df

# To show only the relevant columns and a subset of rows, you can uncomment the following line:
# my_dataset_df[["label", "model_prediction"]][:20]

,text,label,model_prediction
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,NEGATIVE
1,"""I Am Curious: Yellow"" is a risible and preten...",0,NEGATIVE
2,If only to avoid making this type of film in t...,0,NEGATIVE
3,This film was probably inspired by Godard's Ma...,0,POSITIVE
4,"Oh, brother...after hearing about this ridicul...",0,NEGATIVE
...,...,...,...
24995,A hit at the time but now better categorised a...,1,NEGATIVE
24996,I love this movie like no other. Another time ...,1,POSITIVE
24997,This film and it's sequel Barry Mckenzie holds...,1,POSITIVE
24998,'The Adventures Of Barry McKenzie' started lif...,1,NEGATIVE


In [ ]:
review = my_dataset_df.iloc[0]["text"]
print(classifier(review)[0]["label"])

'POSITIVE'

### Using a New model from HuggingFace (FinBERT: for financial analysis using High level helper)

In [ ]:
from transformers import pipeline

finbert_classifier = pipeline("sentiment-analysis", model="ProsusAI/finbert")

In [ ]:
sentence = "The company reported a significant increase in revenue this quarter, exceeding market expectations."
sentence2 = "The company's stock price plummeted after the disappointing earnings report, causing concern among investors."
print(f'Sentence: {sentence}')
print(finbert_classifier(sentence))
print(f'Sentence: {sentence2}')
print(finbert_classifier(sentence2))

[{'label': 'positive', 'score': 0.9554389715194702}]

In [ ]:
sentences = ["Strong consumer demand is driving growth in the retail sector across all regions.",
              "The company's stock price plummeted after the disappointing earnings report, causing concern among investors."]

print(finbert_classifier(sentences))

### Named Entity Recognition

In [ ]:
sentences_ner = ["Apple announced record earnings in the United States on monday.",
                 "I live in New York City and work for Google.",
                 "I am planning to visit the Eiffel Tower in Paris next summer."]

# Standard NER pipeline using a default model (e.g., 'dbmdz/bert-large-cased-finetuned-conll03-english')
ner_pipeline = pipeline("ner")

ner_pipeline(sentences_ner)

### Question Answering